# SahelPay — Partie 3 : Spark SQL et DataFrames

**Objectif** : refaire et enrichir l'analyse de la Partie 2 avec l'API *structurée* (DataFrames et SQL).

**Pré-requis**
- `transactions_history.jsonl` est dans MinIO : `lake/bronze/history/`.
- `merchants.json` est dans MinIO : `lake/bronze/reference/` (à envoyer avec `mcli`, voir les instructions).
- La session de la Partie 2 est **arrêtée** (sinon elle garde les 4 cœurs).

**Ce qu'on fait**
1. Lire les deux fichiers avec un **schéma explicite** (`StructType`), sans inférence.
2. Montant par ville : API DataFrame, puis SQL.
3. Jointure avec les commerçants : montant par catégorie et par ville.
4. UDF `amount_band` contre `when/otherwise`.
5. Écrire un Parquet nettoyé, partitionné par ville, dans MinIO.
6. `explain(True)` sur la jointure : plans logique, optimisé et physique.
7. Comparer RDD et DataFrame avec des temps mesurés.

## 0. Les notions à avoir en tête

### DataFrame, c'est quoi ?
Un **DataFrame** est une table distribuée : des lignes réparties en partitions, mais surtout des **colonnes nommées et typées** (un *schéma*). Sous le capot, c'est toujours des RDD, mais Spark **connaît la structure** des données et peut donc optimiser, ce qu'il ne peut pas faire avec un RDD de dictionnaires Python opaques.

| | RDD (Partie 2) | DataFrame (Partie 3) |
|---|---|---|
| **Contenu** | objets Python quelconques | lignes avec schéma (colonnes typées) |
| **Optimisation** | aucune : Spark exécute exactement ton code | **Catalyst** réécrit et optimise le plan |
| **Exécution** | fonctions Python (lambda) sur les executors | opérations compilées dans la JVM (**Tungsten**) |
| **Style** | « comment » calculer (`map`, `reduceByKey`) | « quoi » calculer (`groupBy`, `sum`, SQL) |

> **Et le Dataset ?** C'est un DataFrame **typé** (`Dataset[Transaction]`) : le compilateur vérifie les types. Il n'existe qu'en **Scala et Java**. En PySpark, un DataFrame est un `Dataset[Row]`, donc il n'y a pas de Dataset typé en Python.

### Pourquoi un schéma explicite (sans inférence) ?
- Avec `inferSchema`, Spark **lit le fichier une première fois** juste pour deviner les types : c'est un passage de plus.
- Le type deviné peut se tromper (un montant `320.0` lu comme entier si aucune ligne n'a de décimale).
- Le schéma explicite est un **contrat** : il garantit les mêmes colonnes et types à chaque lecture, et on le **réutilisera tel quel en Partie 4** pour lire Kafka (`from_json`).

### Catalyst : l'optimiseur
Quand tu écris une requête, Spark passe par 4 étapes (c'est ce que montre `explain(True)`) :
1. **Parsed Logical Plan** : ta requête telle qu'écrite.
2. **Analyzed Logical Plan** : les noms de colonnes et les types sont résolus.
3. **Optimized Logical Plan** : Catalyst réécrit le plan (*predicate pushdown* : filtrer le plus tôt possible, *column pruning* : ne lire que les colonnes utiles, simplification d'expressions).
4. **Physical Plan** : choix des opérateurs réels, par exemple la **stratégie de jointure**.

### Stratégies de jointure
| | BroadcastHashJoin | SortMergeJoin |
|---|---|---|
| **Principe** | la **petite** table est copiée sur chaque executor | les **deux** tables sont redistribuées (shuffle) et triées par clé |
| **Shuffle de la grosse table** | non | oui |
| **Quand** | une table est petite (seuil par défaut : 10 Mo) | les deux tables sont grosses |

Ici le référentiel `merchants` fait 300 lignes (environ 43 Ko) : Spark devrait le diffuser.

## 1. Démarrer la session

In [ ]:
import sys, json, time
sys.path.append("/workspace/scripts")           # pour retrouver spark_session.py
from spark_session import get_spark

from pyspark.sql import functions as F          # fonctions Spark SQL : sum, count, when, col...
from pyspark.sql.types import (StructType, StructField,
                               StringType, DoubleType, TimestampType)

spark = get_spark("SahelPay-Partie3")           # nom visible dans l'UI http://localhost:8080
sc = spark.sparkContext

# Par défaut, un shuffle produit 200 partitions : beaucoup trop pour 12 Mo sur 4 cœurs.
# On le ramène à 8 (2 par cœur) pour éviter des centaines de minuscules tâches.
spark.conf.set("spark.sql.shuffle.partitions", "8")

print("Spark", spark.version, "| master :", sc.master, "| appName :", sc.appName)

## 2. Lire les fichiers avec un schéma explicite

On décrit nous-mêmes chaque colonne et son type avec un `StructType`. Les événements de transactions viennent du fichier `transactions_history.jsonl` ; le référentiel est `merchants.json`.

Remarques :
- `event_time` est lu directement en `TimestampType` : Spark sait interpréter le format ISO `2026-11-15T12:34:56Z`.
- `amount_mad` est un `DoubleType`.
- Tous les champs sont déclarés *nullables* : si une ligne est incomplète, Spark mettra `null` plutôt que de planter.

In [2]:
# Schéma des événements de transaction (une ligne du flux = un changement d'état)
schema_tx = StructType([
    StructField("tx_id",         StringType(),    True),   # identifiant de la transaction
    StructField("type",          StringType(),    True),   # P2M (commerçant) ou P2P (particulier)
    StructField("payer_wallet",  StringType(),    True),   # compte qui paie
    StructField("payee_id",      StringType(),    True),   # M-xxx (commerçant) ou W-xxx (compte)
    StructField("device_id",     StringType(),    True),   # appareil utilisé
    StructField("city",          StringType(),    True),
    StructField("channel",       StringType(),    True),   # APP, QR ou USSD
    StructField("status",        StringType(),    True),   # INITIATED, AUTHORIZED, COMPLETED...
    StructField("amount_mad",    DoubleType(),    True),
    StructField("event_time",    TimestampType(), True),
])

# Schéma du référentiel des commerçants
schema_merchants = StructType([
    StructField("merchant_id", StringType(), True),
    StructField("name",        StringType(), True),
    StructField("city",        StringType(), True),
    StructField("category",    StringType(), True),
    StructField("latitude",    DoubleType(), True),
    StructField("longitude",   DoubleType(), True),
])

In [ ]:
HISTORY_PATH   = "s3a://lake/bronze/history/transactions_history.jsonl"
MERCHANTS_PATH = "s3a://lake/bronze/reference/merchants.json"

t0 = time.perf_counter()

# .schema(...) : on fournit le schéma => Spark n'a PAS besoin de lire le fichier pour deviner les types.
# La lecture est paresseuse : aucune donnée n'est lue tant qu'il n'y a pas d'action.
tx = spark.read.schema(schema_tx).json(HISTORY_PATH)
merchants = spark.read.schema(schema_merchants).json(MERCHANTS_PATH)

tx.cache()                    # on garde en mémoire : on va réutiliser tx pour toutes les questions
merchants.cache()

# ACTION : count() déclenche la lecture réelle et la mise en cache.
nb_tx = tx.count()
nb_m = merchants.count()
temps_df = {"chargement (à froid)": time.perf_counter() - t0}

print(f"Événements lus : {nb_tx:,} | Commerçants : {nb_m}   (en {temps_df['chargement (à froid)']:.2f} s)\n")
tx.printSchema()
tx.show(5, truncate=False)
merchants.show(5, truncate=False)

26/11/26 22:20:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
[Stage 0:>                                                          (0 + 0) / 3]

## 3. Montant par ville : API DataFrame, puis SQL

On refait le calcul de la Partie 2 de deux façons qui donnent **exactement le même plan d'exécution**, car Catalyst les ramène à la même chose :
- **API DataFrame** : `filter`, `groupBy`, `agg`.
- **SQL** : on enregistre le DataFrame comme *vue temporaire* (`createOrReplaceTempView`) et on l'interroge avec `spark.sql(...)`.

On calcule en même temps le total, le nombre de transactions et la moyenne (en un seul passage, comme en Partie 2).

In [ ]:
# --- Version API DataFrame ------------------------------------------------------
completed_df = tx.filter(F.col("status") == "COMPLETED")        # garde les transactions COMPLETED

total_df = (completed_df
    .groupBy("city")                                             # un groupe par ville
    .agg(F.sum("amount_mad").alias("total_mad"),                 # somme des montants
         F.count("*").alias("nb_transactions"),                  # nombre de transactions
         F.avg("amount_mad").alias("moyenne_mad"))               # moyenne
    .orderBy(F.desc("total_mad")))                               # tri décroissant

total_df.select("city",
                F.round("total_mad", 2).alias("total_mad"),
                "nb_transactions",
                F.round("moyenne_mad", 2).alias("moyenne_mad")).show()

In [ ]:
# --- Version SQL ----------------------------------------------------------------
# Une "vue temporaire" donne un nom SQL à un DataFrame (elle existe le temps de la session).
tx.createOrReplaceTempView("transactions")
merchants.createOrReplaceTempView("merchants")

total_sql = spark.sql("""
    SELECT city,
           ROUND(SUM(amount_mad), 2)  AS total_mad,
           COUNT(*)                   AS nb_transactions,
           ROUND(AVG(amount_mad), 2)  AS moyenne_mad
    FROM transactions
    WHERE status = 'COMPLETED'
    GROUP BY city
    ORDER BY total_mad DESC
""")
total_sql.show()

# Vérification : les deux versions doivent donner exactement les mêmes lignes.
ref = total_df.select("city", F.round("total_mad", 2), "nb_transactions", F.round("moyenne_mad", 2)).collect()
print("API DataFrame et SQL identiques :", [tuple(r) for r in ref] == [tuple(r) for r in total_sql.collect()])

Les résultats doivent être **identiques à ceux de la Partie 2** (même total par ville, même nombre de transactions, même moyenne). Si ce n'est pas le cas, c'est qu'il y a un problème de lecture ou de schéma.

## 4. Jointure avec le référentiel des commerçants

On veut le **montant encaissé par catégorie de commerçant et par ville**.
- On ne garde que les paiements chez un commerçant (`type = 'P2M'`) **validés** (`COMPLETED`).
- On joint `payee_id` (côté transaction) avec `merchant_id` (côté référentiel).
- Les deux tables ont une colonne `city` : on renomme celle du référentiel en `merchant_city` pour éviter l'ambiguïté.

Avant de calculer, une **vérification de qualité** : combien de paiements pointent vers un commerçant inconnu ? Un `left_anti` join renvoie les lignes de gauche **sans correspondance** à droite. On attend 0.

In [ ]:
# Référentiel allégé : on ne garde que les colonnes utiles, et on renomme "city" pour éviter l'ambiguïté.
m = merchants.select("merchant_id", F.col("city").alias("merchant_city"), "category")

# Paiements commerçants validés
p2m = completed_df.filter(F.col("type") == "P2M")

# Contrôle qualité : paiements dont le commerçant n'existe pas dans le référentiel (attendu : 0)
orphelins = p2m.join(m, p2m.payee_id == m.merchant_id, "left_anti").count()
print("Paiements vers un commerçant inconnu :", orphelins)

# Jointure interne : chaque paiement reçoit la catégorie et la ville de son commerçant.
joined = p2m.join(m, p2m.payee_id == m.merchant_id, "inner")

# Montant encaissé par catégorie et par ville
par_cat_ville = (joined
    .groupBy("category", "merchant_city")
    .agg(F.round(F.sum("amount_mad"), 2).alias("total_mad"),
         F.count("*").alias("nb_paiements"))
    .orderBy("category", F.desc("total_mad")))

print("Lignes (catégorie x ville) :", par_cat_ville.count())
par_cat_ville.show(20, truncate=False)

In [ ]:
# Même résultat en tableau croisé : une ligne par ville, une colonne par catégorie.
# pivot() transforme les valeurs de "category" en colonnes (plus lisible pour un rapport).
(joined.groupBy("merchant_city")
       .pivot("category")
       .agg(F.round(F.sum("amount_mad"), 0))
       .orderBy("merchant_city")
       .show(truncate=False))

## 5. UDF `amount_band` contre `when/otherwise`

On classe chaque montant en trois tranches :
- `micro` : moins de 50 MAD ;
- `standard` : de 50 à 2 000 MAD inclus ;
- `eleve` : plus de 2 000 MAD.

On l'écrit **deux fois** pour comparer :
1. une **UDF** (*User Defined Function*) : une fonction Python que Spark appelle pour chaque ligne ;
2. des expressions Spark natives `when / otherwise`.

### Pourquoi la version native est plus rapide
- **UDF Python** : pour chaque ligne, Spark doit **sérialiser** la valeur, l'envoyer de la JVM vers un processus Python, exécuter la fonction, puis renvoyer le résultat. Pour Catalyst, l'UDF est une **boîte noire** : il ne peut ni l'optimiser, ni la compiler.
- **`when/otherwise`** : c'est une expression que Catalyst comprend (`CASE WHEN`). Elle est compilée en code JVM (*whole-stage codegen*) et exécutée **sans quitter la JVM**, sans aller-retour Python.

In [ ]:
# --- Version 1 : UDF Python -----------------------------------------------------
def amount_band(amount_mad):
    """Retourne la tranche d'un montant : micro (<50), standard (50 à 2000), eleve (>2000)."""
    if amount_mad is None:          # un montant manquant n'a pas de tranche
        return None
    if amount_mad < 50:
        return "micro"
    if amount_mad <= 2000:
        return "standard"
    return "eleve"

# F.udf(...) : fonction utilisable avec l'API DataFrame.
amount_band_udf = F.udf(amount_band, StringType())
# spark.udf.register(...) : même fonction, utilisable dans les requêtes SQL (amount_band(...)).
spark.udf.register("amount_band", amount_band, StringType())

# --- Version 2 : expression native when / otherwise -----------------------------
band_expr = (F.when(F.col("amount_mad").isNull(), F.lit(None))   # montant manquant -> pas de tranche
              .when(F.col("amount_mad") < 50, "micro")
              .when(F.col("amount_mad") <= 2000, "standard")
              .otherwise("eleve"))

# Test rapide des deux versions sur quelques lignes
tx.select("tx_id", "amount_mad",
          amount_band_udf("amount_mad").alias("band_udf"),
          band_expr.alias("band_native")).show(5)

# Usage en SQL
spark.sql("SELECT amount_mad, amount_band(amount_mad) AS band FROM transactions LIMIT 3").show()

In [ ]:
# --- Comparaison de performance ---------------------------------------------------
def chronometrer(calcul, n=3):
    """Exécute 'calcul' n fois et renvoie la MÉDIANE des durées (plus stable qu'une seule mesure)."""
    durees = []
    for _ in range(n):
        t0 = time.perf_counter()
        calcul()
        durees.append(time.perf_counter() - t0)
    return sorted(durees)[n // 2]

# Même calcul des deux côtés : on calcule la tranche de TOUTES les lignes puis on compte par tranche.
# groupBy + count force Spark à évaluer la colonne sur chaque ligne.
repartition_udf    = lambda: tx.withColumn("band", amount_band_udf("amount_mad")).groupBy("band").count().collect()
repartition_native = lambda: tx.withColumn("band", band_expr).groupBy("band").count().collect()

# Les deux versions doivent donner les mêmes comptes.
res_udf    = sorted((r["band"], r["count"]) for r in repartition_udf())
res_native = sorted((r["band"], r["count"]) for r in repartition_native())
print("Même résultat :", res_udf == res_native, "->", res_native)

temps_df["amount_band UDF (médiane)"]    = chronometrer(repartition_udf)
temps_df["amount_band natif (médiane)"] = chronometrer(repartition_native)
print(f"UDF Python        : {temps_df['amount_band UDF (médiane)']:.3f} s")
print(f"when / otherwise  : {temps_df['amount_band natif (médiane)']:.3f} s")
print(f"Rapport UDF / natif : x{temps_df['amount_band UDF (médiane)'] / temps_df['amount_band natif (médiane)']:.1f}")

In [ ]:
# Dans le plan physique, l'UDF apparaît comme un opérateur Python séparé (BatchEvalPython),
# alors que la version native reste une simple expression CASE WHEN dans la JVM.
print("=== Plan avec UDF ===")
tx.withColumn("band", amount_band_udf("amount_mad")).groupBy("band").count().explain()
print("\n=== Plan natif ===")
tx.withColumn("band", band_expr).groupBy("band").count().explain()

## 6. Écrire le résultat nettoyé en Parquet, partitionné par ville

**Parquet** est un format **colonnaire** compressé : on ne lit que les colonnes dont on a besoin, et les types sont conservés (contrairement au JSON).

**Partitionner par `city`** crée un sous-dossier par ville (`city=Rabat/`, `city=Fes/`...). Une requête qui filtre sur une ville ne lit que son dossier : c'est le *partition pruning*.

Ce qu'on entend par « nettoyé » ici :
- **doublons supprimés** sur `(tx_id, status)` (le générateur en injecte environ 2 %) ;
- `event_time` typé en **timestamp** (déjà fait par le schéma) ;
- colonne **`amount_band`** ajoutée (version native, plus performante).

On garde **tous les statuts** : filtrer sur `COMPLETED` se fera au moment des analyses.

`repartition("city")` regroupe d'abord toutes les lignes d'une même ville dans la même partition, pour obtenir peu de fichiers par ville au lieu d'un petit fichier par ville *et par partition*.

In [ ]:
SILVER_PATH = "s3a://lake/silver/transactions_history/"

silver = (tx
    .dropDuplicates(["tx_id", "status"])            # un seul événement par (transaction, statut)
    .withColumn("amount_band", band_expr))          # colonne tranche de montant (when/otherwise)

(silver
    .repartition("city")                            # regroupe les lignes d'une ville ensemble
    .write
    .mode("overwrite")                              # remplace si ça existe déjà (relançable)
    .partitionBy("city")                            # un sous-dossier par ville
    .parquet(SILVER_PATH))

# Relecture pour vérifier : Spark retrouve la colonne "city" grâce aux noms de dossiers.
relu = spark.read.parquet(SILVER_PATH)
print(f"Lignes avant nettoyage : {tx.count():,}")
print(f"Lignes dans le Parquet : {relu.count():,}   (doublons retirés : {tx.count() - relu.count():,})")
print("Fichiers Parquet écrits :", len(relu.inputFiles()))
relu.groupBy("city").count().orderBy(F.desc("count")).show()
relu.printSchema()

Dans MinIO, tu dois voir `lake/silver/transactions_history/` avec un dossier `city=...` par ville. Pour la capture du rapport (`mc ls`), lance dans PowerShell :

```powershell
docker exec minio mcli ls --recursive local/lake/silver/
```

## 7. `explain(True)` sur la jointure

On lit le plan de la requête « paiements validés ⨝ commerçants ».

> **Piège du cache** : Spark remplace automatiquement toute lecture *identique* à un DataFrame déjà mis en cache par une lecture du cache. Si on laisse `tx` et `merchants` en cache, le plan affiche `InMemoryTableScan` au lieu de `FileScan`, et on ne voit plus ni la lecture du fichier ni les filtres poussés vers la source. On **vide donc le cache** (`unpersist`) avant d'afficher le plan. Ça garantit aussi que la section 8 mesure une vraie lecture du fichier.

### Comment lire la sortie
`explain(True)` affiche **quatre blocs** :
1. `== Parsed Logical Plan ==` : la requête brute.
2. `== Analyzed Logical Plan ==` : colonnes et types résolus.
3. `== Optimized Logical Plan ==` : après Catalyst. Compare avec le plan analysé : Catalyst a **ajouté des filtres `isnotnull(...)`** sur les clés de jointure (une valeur nulle ne peut jamais correspondre), et le `Filter` est collé à la source (*predicate pushdown*). Dans le plan physique, côté `merchants`, le `FileScan` ne lit que **3 colonnes sur 6** (*column pruning*).
4. `== Physical Plan ==` : les vrais opérateurs. Repère la **stratégie de jointure** : `BroadcastHashJoin` ou `SortMergeJoin`, et `BroadcastExchange` (la diffusion de la petite table).

> Avec l'*Adaptive Query Execution* (activée par défaut dans Spark 3.5), le plan physique commence par `AdaptiveSparkPlan isFinalPlan=false` : Spark peut encore le modifier pendant l'exécution.

In [ ]:
# On vide le cache : sinon Spark réutiliserait les DataFrames en mémoire et le plan
# n'afficherait plus la lecture des fichiers (voir l'encadré ci-dessus).
tx.unpersist()
merchants.unpersist()

def requete_jointure():
    """Construit la requête sur des lectures NEUVES (sans cache) pour un plan lisible."""
    tx_nc = spark.read.schema(schema_tx).json(HISTORY_PATH)
    m_nc = (spark.read.schema(schema_merchants).json(MERCHANTS_PATH)
            .select("merchant_id", F.col("city").alias("merchant_city"), "category"))
    paiements = tx_nc.filter((F.col("status") == "COMPLETED") & (F.col("type") == "P2M"))
    return paiements.join(m_nc, paiements.payee_id == m_nc.merchant_id, "inner")

# explain(True) : les 4 plans (parsed, analyzed, optimized, physical)
requete_jointure().explain(True)

### Forcer l'autre stratégie : SortMergeJoin

Pour comprendre la différence, on **interdit la diffusion** (`autoBroadcastJoinThreshold = -1`) et on regarde le plan : Spark doit alors utiliser un `SortMergeJoin`, avec un `Exchange` (shuffle) et un `Sort` **des deux côtés**. On refait un DataFrame après le changement de configuration, car le plan est fixé à la création de la requête.

In [ ]:
# On interdit le broadcast : plus aucune table n'est assez "petite" pour être diffusée.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("=== Plan physique SANS broadcast ===")
requete_jointure().explain()                         # on attend SortMergeJoin

# On remet le seuil par défaut (10 Mo).
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
print("\n=== Plan physique AVEC broadcast (réglage par défaut) ===")
requete_jointure().explain()                         # on attend BroadcastHashJoin

## 8. Comparer RDD et DataFrame avec des temps mesurés

Le rapport demande de comparer RDD, DataFrame et Dataset **avec des temps mesurés**. Pour que la comparaison soit honnête, on refait **le même calcul** dans **la même session**, avec le même protocole :
- chargement + parsing de 50 000 événements ;
- calcul du total par ville, **médiane de 3 exécutions « à chaud »**, données en cache des deux côtés.

> Les temps « à froid » de la Partie 2 (par exemple 34 s de chargement) incluent le démarrage des processus Python et la première connexion à MinIO : ils ne sont **pas** comparables à un temps à chaud. D'où cette section.

In [ ]:
from operator import add     # opérateur "+" sous forme de fonction, pour reduceByKey

# --- Côté RDD : lecture + parsing en Python (json.loads sur chaque ligne) ------------
t0 = time.perf_counter()
ev_rdd = sc.textFile(HISTORY_PATH, minPartitions=4).map(json.loads)
ev_rdd.cache()
ev_rdd.count()                                                   # déclenche lecture + parsing + cache
t_load_rdd = time.perf_counter() - t0

# --- Côté DataFrame : lecture JSON dans la JVM avec schéma explicite -------------------
t0 = time.perf_counter()
ev_df = spark.read.schema(schema_tx).json(HISTORY_PATH)
ev_df.cache()
ev_df.count()
t_load_df = time.perf_counter() - t0

# --- Le même calcul : total par ville des transactions COMPLETED ---------------------
total_rdd = lambda: (ev_rdd.filter(lambda e: e["status"] == "COMPLETED")
                           .map(lambda e: (e["city"], e["amount_mad"]))
                           .reduceByKey(add).collect())

total_dfq = lambda: (ev_df.filter(F.col("status") == "COMPLETED")
                          .groupBy("city").agg(F.sum("amount_mad")).collect())

# Les deux doivent donner les mêmes totaux (à l'arrondi près).
a = dict(total_rdd()); b = {r["city"]: r["sum(amount_mad)"] for r in total_dfq()}
print("Mêmes totaux RDD et DataFrame :", all(abs(a[v] - b[v]) < 1e-6 for v in a))

t_total_rdd = chronometrer(total_rdd)
t_total_df  = chronometrer(total_dfq)

print(f"{'Étape':<34} {'RDD':>9} {'DataFrame':>11}")
print("-" * 56)
print(f"{'Chargement + parsing (2e lecture)':<34} {t_load_rdd:>8.2f}s {t_load_df:>10.2f}s")
print(f"{'Total par ville (médiane de 3)':<34} {t_total_rdd:>8.3f}s {t_total_df:>10.3f}s")

### Comment interpréter

- **Chargement** : le DataFrame lit le JSON dans la **JVM** (parser natif), alors que le RDD appelle `json.loads` en **Python** pour chaque ligne.
- **Calcul** : le DataFrame bénéficie de Catalyst (par exemple la lecture des seules colonnes utiles) et de Tungsten (code JVM compilé). Le RDD exécute des lambdas Python.
- **Sur 12 Mo, les écarts de calcul peuvent être faibles**, car le temps est dominé par la coordination du cluster. L'écart se creuse avec des volumes plus grands, et l'écart de l'UDF (section 5) montre bien le coût d'un aller-retour Python.
- **Dataset** : même moteur que le DataFrame, mais typé à la compilation, disponible seulement en Scala/Java. En PySpark, on n'en a pas besoin.

## 9. Récapitulatif pour le rapport

In [ ]:
print(f"{'Mesure (DataFrame / Partie 3)':<36} {'Temps (s)':>10}")
print("-" * 48)
for etape, secondes in temps_df.items():
    print(f"{etape:<36} {secondes:>10.3f}")

### Captures à faire **avant d'arrêter la session**
1. **http://localhost:4040 → Jobs** : un job de la jointure ou de l'agrégation. Ouvre-le et capture la **DAG Visualization** et le nombre de stages.
2. **http://localhost:4040 → SQL / DataFrame** : ouvre une requête (par exemple la jointure) pour voir le **plan physique en graphe**. C'est la meilleure capture pour montrer Catalyst et `BroadcastHashJoin`.
3. La **sortie d'`explain(True)`** (section 7) : capture les 4 blocs, en entier.
4. **MinIO** : `docker exec minio mcli ls --recursive local/lake/silver/` (Parquet partitionné par ville).
5. **http://localhost:8080** avec `SahelPay-Partie3` en cours.

### À rédiger dans le rapport (avec tes mots)
- **`explain(True)`** : identifie le plan logique, le plan optimisé par Catalyst (ce qui a changé : filtre poussé vers la source, colonnes élaguées) et la stratégie de jointure choisie, avec la raison (table de 300 lignes sous le seuil de 10 Mo).
- **UDF contre `when/otherwise`** : pourquoi la version native est plus rapide (sérialisation JVM ↔ Python, boîte noire pour Catalyst, absence de codegen), chiffres à l'appui.
- **RDD, DataFrame, Dataset** : typage, optimisation (Catalyst / Tungsten), et tes temps mesurés (section 8). N'oublie pas que le Dataset typé n'existe qu'en Scala/Java.
- **Schéma explicite** : pourquoi il vaut mieux que l'inférence.
- **Parquet partitionné** : format colonnaire, partition pruning.

## 10. Libérer le cluster

> À exécuter **seulement après** les captures.

In [ ]:
# Arrête la session et libère les 4 cœurs.
# spark.stop()